# 🏷️ NB-13: Named Entity Recognition – Extract Concepts from Claude
**Goal:** Fine-tune a token classification model to recognize technical concepts, tools, algorithms, and proper nouns in Claude's responses.

**Modality:** Token Classification / NER | **Model:** dslim/bert-base-NER

In [ ]:
!pip install transformers datasets seqeval -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import load_dataset
import numpy as np
import evaluate

# Use standard CoNLL-2003 as base, adapt with Claude vocabulary
conll = load_dataset("conll2003", trust_remote_code=True)
label_list = conll["train"].features["ner_tags"].feature.names
print("NER labels:", label_list)

MODEL = "dslim/bert-base-NER"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(label_list))

def tokenize_and_align(batch):
    enc = tok(batch["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(batch["ner_tags"]):
        word_ids = enc.word_ids(batch_index=i)
        prev = None; aligned = []
        for wid in word_ids:
            if wid is None: aligned.append(-100)
            elif wid != prev: aligned.append(label[wid])
            else: aligned.append(-100)
            prev = wid
        labels.append(aligned)
    enc["labels"] = labels
    return enc

tokenized = conll.map(tokenize_and_align, batched=True, remove_columns=conll["train"].column_names)
seqeval = evaluate.load("seqeval")

def metrics(p):
    preds = np.argmax(p.predictions, axis=2)
    true_p = [[label_list[p] for p,l in zip(pred,lab) if l != -100] for pred,lab in zip(preds, p.label_ids)]
    true_l = [[label_list[l] for _,l in zip(pred,lab) if l != -100] for pred,lab in zip(preds, p.label_ids)]
    r = seqeval.compute(predictions=true_p, references=true_l)
    return {"f1": r["overall_f1"], "precision": r["overall_precision"], "recall": r["overall_recall"]}

args = TrainingArguments("./ner-claude", num_train_epochs=3, per_device_train_batch_size=16,
    fp16=True, evaluation_strategy="epoch", report_to="none")
Trainer(model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForTokenClassification(tok), compute_metrics=metrics).train()
print("✅ NER model fine-tuned!")


In [ ]:
# --- Annotate Claude responses with NER ---
from transformers import pipeline
ner = pipeline("ner", model=model, tokenizer=tok, aggregation_strategy="simple")
sample = data[0]["response"][:300]
entities = ner(sample)
for e in entities[:10]:
    print(f"{e['entity_group']:10s} | {e['score']:.2f} | {e['word']}")
